# Solid fuels model — monthly BVAR-SV-outlier


> **Data contract.** This notebook reads only the model-ready CSV under `data/processed/<vintage>/`. The single raw Excel workbook is handled upstream by the dataset builder; changing source tickers does not require notebook code changes as long as the processed canonical columns remain available.

This notebook estimates the solid fuels component of the energy-inflation suite. The endogenous block contains monthly absolute changes in wholesale natural-gas prices, PPI Energy and the HICP solid fuels index. The model uses 12 lags and no deterministic seasonal regressors.

## 1. Imports and configuration


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "src" / "model",
    PROJECT_ROOT,
    Path.cwd(),
]
for candidate in MODEL_DIR_CANDIDATES:
    required_modules = (
        "energy_bvar_model.py",
        "energy_bvar_monthly_hicp.py",
        "energy_bvar_plot.py",
        "energy_bvar_io.py",
    )
    if all((candidate / name).exists() for name in required_modules):
        sys.path.insert(0, str(candidate))
        MODEL_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "Place energy_bvar_model.py, energy_bvar_monthly_hicp.py, "
        "energy_bvar_plot.py and energy_bvar_io.py beside this notebook "
        "or in src/model/."
    )

from energy_bvar_model import (
    BVARSVOPriorConfig,
    SamplerConfig,
    coefficient_posterior_tables,
    forecast_bvar_sv_outlier,
    forecast_error_variance_decomposition,
    forecast_horizon_table,
    historical_decomposition,
    impulse_responses,
    make_bvar_svo_prior,
    mcmc_diagnostics,
    nowcast_summary_table,
    path_horizon_table,
    posterior_outlier_probability,
    posterior_predictive_statistics,
    prepare_bvar_panel,
    residual_diagnostic_table,
    run_energy_bvar,
    stability_diagnostics,
    stationarity_tests,
)
from energy_bvar_io import save_energy_bvar_result
from energy_bvar_monthly_hicp import (
    forecast_hicp_component,
    load_monthly_hicp_component_panel,
    monthly_hicp_component_spec,
    monthly_hicp_series_construction_table,
)
from energy_bvar_plot import (
    format_number,
    plot_absolute_differences,
    plot_chain_acf_grid,
    plot_coefficient_heatmap,
    plot_data_levels,
    plot_fevd,
    plot_forecast_comparison,
    plot_hicp_forecast_fan,
    plot_historical_decomposition,
    plot_impulse_responses,
    plot_minnesota_prior_by_equation,
    plot_nowcast_fan,
    plot_phi_and_outlier_priors,
    plot_posterior_predictive_check,
    plot_short_forecast_fan,
    plot_shrinkage_comparison,
    plot_spectral_radius,
    plot_standardized_residuals,
    plot_trace_grid,
    plot_volatility_decomposition,
    style_numeric_table,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

COMPONENT = "solid_fuels"
SPEC = monthly_hicp_component_spec(COMPONENT)
SEED = 42
P = 12
MISSING_DATA_METHOD = "linear"  # "linear" = fast approximation; "dk" = exact interior-missing treatment
SIMULATE_FUTURE_OUTLIERS = True
CODE_VERSION = "energy_bvar_model-v2"
VARIABLES = SPEC["variables"]
TARGET = SPEC["target"]
UNITS = SPEC["units"]
SAVE_RESULTS = False
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Model modules: {MODEL_DIR}")
print(f"Component: {COMPONENT}")

## 2. Data


### Missing-data treatment used in this notebook

`MISSING_DATA_METHOD = "linear"` is the fast default. Bounded interior missing **levels** are interpolated once; these pseudo-observations then enter the likelihood as fixed values. Set `MISSING_DATA_METHOD = "dk"` to propagate uncertainty about interior gaps through the Gibbs sampler. In both modes, the ragged edge and future conditional paths are handled by Durbin–Koopman state-space simulation. The result metadata and forecast-table/plot labels surface whether an approximation was actually used.


In [ ]:
# Set VINTAGE = "YYYYMMDD" to force one processed vintage.
VINTAGE = None
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

if VINTAGE is None:
    candidates = sorted(
        path for path in PROCESSED_ROOT.glob("*/solid_fuels_monthly.csv")
        if path.is_file()
    )
    if not candidates:
        raise FileNotFoundError(
            f"No solid_fuels_monthly.csv found below {PROCESSED_ROOT}."
        )
    COMPONENT_DATASET = candidates[-1]
else:
    COMPONENT_DATASET = PROCESSED_ROOT / VINTAGE / "solid_fuels_monthly.csv"

levels = load_monthly_hicp_component_panel(
    COMPONENT_DATASET,
    component=COMPONENT,
    variables=VARIABLES,
)

print(f"Dataset: {COMPONENT_DATASET}")
print(f"Sample calendar: {levels.index.min().date()} to {levels.index.max().date()}")

display(style_numeric_table(
    monthly_hicp_series_construction_table(COMPONENT_DATASET, COMPONENT),
    caption=f"Construction of the {SPEC['label']}-model level series",
))
display(style_numeric_table(
    levels.tail(12),
    caption="Latest processed levels",
))
plot_data_levels(levels, units=UNITS)
plt.show()

### Absolute monthly changes

The endogenous vector is ordered from upstream market prices to the consumer-price target:

$$
y_t =
\begin{bmatrix}
\Delta P_t^{\mathrm{gas}} \\
\Delta P_t^{\mathrm{PPI\ energy}} \\
\Delta HICP_t^{\mathrm{solid fuels}}
\end{bmatrix},
\qquad
\Delta x_t=x_t-x_{t-1}.
$$

All three variables remain in absolute differences rather than log differences.


In [ ]:
prep = prepare_bvar_panel(
    levels,
    p=P,
    variables=VARIABLES,
    missing_data_method=MISSING_DATA_METHOD,
)

transformation_summary = pd.Series({
    "missing_data_method": prep["missing_data_method"],
    "missing_treatment_exact": prep["missing_treatment_exact"],
    "interpolated_level_cells": prep["n_interpolated_level_cells"],
    "balanced_change_start": prep["balanced_start"],
    "balanced_change_end": prep["balanced_end"],
    "balanced_changes": prep["n_balanced_changes"],
    "VAR_regression_observations": prep["n_regression_observations"],
    "regression_columns_per_equation": prep["k"],
    "ragged_edge_months": prep["n_ragged_months"],
    "latest_calendar_month": prep["last_calendar_date"],
})
display(style_numeric_table(
    transformation_summary.to_frame("value"),
    caption="Transformation and estimation sample",
))
display(style_numeric_table(
    prep["ragged"],
    caption="Ragged-edge absolute changes",
))

if prep.get("n_interpolated_level_cells", 0):
    linear_imputation = (
        prep["levels"]
        .where(prep["interpolated_level_mask"])
        .stack()
        .rename("linear_imputation")
        .to_frame()
    )
    linear_imputation.index.names = ["date", "variable"]
    display(style_numeric_table(
        linear_imputation.head(30),
        caption="Linear interpolation audit — first imputed interior levels",
    ))

plot_absolute_differences(prep["differences"], units=UNITS)
plt.show()

## 3. Model

For $t=1,\ldots,T$, the model is

$$
y_t = c + \sum_{\ell=1}^{12} B_\ell y_{t-\ell} + \nu_t.
$$

The residual covariance follows

$$
\nu_t=A^{-1}O_t\Lambda_t^{1/2}\varepsilon_t,
\qquad \varepsilon_t\sim\mathcal N(0,I),
$$

where the diagonal elements of $\Lambda_t$ follow independent log random walks and $O_t$ contains the transitory outlier scales.


## 4. Priors


In [ ]:
prior_config = BVARSVOPriorConfig(
    lambda1=0.20,
    lambda2=0.50,
    lambda3=1.00,
    lambda4=10.0,
    a_prior_var=10.0,
    phi_prior_mean=0.02,
    phi_prior_df=10.0,
    h0_var=4.0,
    outlier_mean_frequency=1.0 / 48.0,
    outlier_prior_observations=120.0,
    outlier_grid_min=2.0,
    outlier_grid_max=20.0,
    outlier_grid_step=1.0,
    ksc_offset_scale=1e-6,
)
prior = make_bvar_svo_prior(prep, prior_config)

prior_table = pd.DataFrame({
    "value": [
        prior_config.lambda1,
        prior_config.lambda2,
        prior_config.lambda3,
        prior_config.lambda4,
        prior_config.a_prior_var,
        prior_config.phi_prior_mean,
        prior["outlier_alpha"],
        prior["outlier_beta"],
    ],
    "interpretation": [
        "overall Minnesota lag tightness",
        "additional cross-lag tightness",
        "lag-decay exponent",
        "constant prior scale",
        "variance of each free contemporaneous A coefficient",
        "prior mean of volatility-of-volatility variance",
        "Beta outlier pseudo-count",
        "Beta regular-observation pseudo-count",
    ],
}, index=[
    "lambda1", "lambda2", "lambda3", "lambda4",
    "A prior variance", "phi prior mean",
    "outlier alpha", "outlier beta",
])
display(style_numeric_table(prior_table, caption="Prior hyperparameters"))

prior_figures = plot_minnesota_prior_by_equation(prior, VARIABLES, P)
for equation, figure in prior_figures.items():
    display(figure)
    plt.close(figure)
plot_phi_and_outlier_priors(prior)
plt.show()

## 5. Posterior and Gibbs sampler

The coefficient block is drawn from its heteroskedastic Gaussian conditional and truncated to stable VAR dynamics. The contemporaneous matrix, complete stochastic-volatility paths, volatility innovation variances, outlier states and outlier probabilities are then updated conditionally inside each Gibbs iteration.


In [ ]:
sampler_config = SamplerConfig(
    reps=6_000,
    burn=3_000,
    thin=1,
    seed=SEED,
    max_stability_tries=1_000,
    progress_every=500,
)

result = run_energy_bvar(
    levels=levels,
    model_id=COMPONENT,
    vintage=COMPONENT_DATASET.parent.name,
    p=P,
    variables=VARIABLES,
    prior_config=prior_config,
    sampler_config=sampler_config,
    missing_data_method=MISSING_DATA_METHOD,
    code_version=CODE_VERSION,
)

print(f"Retained posterior draws: {result['n_draws']:,}")
print(f"Run ID: {result['metadata']['run_id']}")
print(f"Missing-data method: {result['metadata']['missing_data_method']}")
print(f"Exact missing-data treatment: {result['metadata']['missing_treatment_exact']}")
print(f"Interpolated level cells: {result['metadata']['interpolated_level_cells']}")
print(f"B instability proposal rejection rate: {result['B_instability_rejection_rate']:.2%}")
display(style_numeric_table(
    pd.Series(dict(zip(VARIABLES, result["ksc_offsets"]))).to_frame("KSC offset"),
    caption="Adaptive KSC offsets",
))

if SAVE_RESULTS:
    saved_result_paths = save_energy_bvar_result(result, RESULTS_ROOT)
    print(f"Saved run: {saved_result_paths['directory']}")

## 6. Results


### 6.1 Posterior coefficients


In [ ]:
coefficient_tables = coefficient_posterior_tables(result)
for equation in VARIABLES:
    print()
    print(f"{equation} equation")
    display(style_numeric_table(
        coefficient_tables[equation],
        caption=f"Posterior coefficients — {equation} equation",
    ))
    plot_coefficient_heatmap(result, equation=equation)
    plot_shrinkage_comparison(result, equations=[equation])
    plt.show()

### 6.2 Stochastic volatility and transitory outliers


In [ ]:
outlier_probability = posterior_outlier_probability(result)
important_outliers = (
    outlier_probability.stack()
    .rename("posterior_probability")
    .loc[lambda series: series >= 0.50]
    .sort_values(ascending=False)
    .to_frame()
)
display(style_numeric_table(
    important_outliers.head(30),
    caption="Dates with posterior outlier probability at least 50%",
))
plot_volatility_decomposition(result)
plt.show()

### 6.3 Standardized structural residuals


In [ ]:
display(style_numeric_table(
    residual_diagnostic_table(result),
    caption="Standardized structural-residual diagnostics",
))
plot_standardized_residuals(result)
plt.show()

## 7. Ragged-edge nowcast and forecasts


In [ ]:
FORECAST_ORIGIN = prep["last_calendar_date"]
MONTHS_TO_YEAR_END = 12 - FORECAST_ORIGIN.month
if MONTHS_TO_YEAR_END == 0:
    MONTHS_TO_YEAR_END = 12
H = max(3, MONTHS_TO_YEAR_END)
FORECAST_DRAWS = min(1_000, result["n_draws"])

forecast_unconditional = forecast_bvar_sv_outlier(
    result,
    H=H,
    n_draws=FORECAST_DRAWS,
    simulate_future_outliers=SIMULATE_FUTURE_OUTLIERS,
    seed=2026,
)

print(f"Forecast origin: {FORECAST_ORIGIN.date()}")
print(f"Forecast horizon used: {H} months")

nowcast_table = nowcast_summary_table(
    forecast_unconditional,
    observed_levels=levels,
    variable=TARGET,
)
if nowcast_table.empty:
    print(f"No missing {TARGET} observation is present in the ragged edge.")
else:
    display(style_numeric_table(
        nowcast_table,
        caption=f"Ragged-edge nowcast — {TARGET}",
    ))
plot_nowcast_fan(
    forecast_unconditional,
    historical_levels=levels,
    variable=TARGET,
    ylabel=UNITS[TARGET],
)
plt.show()

In [ ]:
for variable in VARIABLES:
    table = forecast_horizon_table(forecast_unconditional, variable)
    display(style_numeric_table(
        table,
        caption=f"Unconditional level forecast — {variable} ({UNITS[variable]})",
    ))
    plot_short_forecast_fan(
        forecast_unconditional,
        historical_levels=levels,
        variable=variable,
        ylabel=UNITS[variable],
    )
    plt.show()

### 7.1 Conditional wholesale-gas scenario

The future wholesale natural-gas level is fixed at 10% above its latest observed value. This is a scenario comparison; the other variables remain unconstrained.


In [ ]:
last_wholesale_level = levels["natural_gas_wholesale"].dropna().iloc[-1]
wholesale_up_10 = np.full(H, 1.10 * last_wholesale_level)

forecast_gas_up_10 = forecast_bvar_sv_outlier(
    result,
    H=H,
    level_conditions={"natural_gas_wholesale": wholesale_up_10},
    n_draws=FORECAST_DRAWS,
    simulate_future_outliers=SIMULATE_FUTURE_OUTLIERS,
    seed=2026,
)

condition_error = np.max(np.abs(
    forecast_gas_up_10["future_level_paths"][:, :, VARIABLES.index("natural_gas_wholesale")]
    - wholesale_up_10[None, :]
))
print(f"Maximum absolute wholesale-gas condition error: {format_number(condition_error)}")

display(style_numeric_table(
    forecast_horizon_table(forecast_gas_up_10, TARGET),
    caption=f"Conditional {TARGET} forecast — wholesale gas +10%",
))
plot_forecast_comparison(
    forecast_unconditional,
    forecast_gas_up_10,
    historical_levels=levels,
    variable=TARGET,
    ylabel=UNITS[TARGET],
)
plt.show()

### 7.2 Conditional PPI Energy scenario

The future PPI Energy level is fixed at 10% above its latest observed value, while wholesale gas and the HICP target remain unconstrained.


In [ ]:
last_ppi_level = levels["ppi_energy"].dropna().iloc[-1]
ppi_up_10 = np.full(H, 1.10 * last_ppi_level)

forecast_ppi_up_10 = forecast_bvar_sv_outlier(
    result,
    H=H,
    level_conditions={"ppi_energy": ppi_up_10},
    n_draws=FORECAST_DRAWS,
    simulate_future_outliers=SIMULATE_FUTURE_OUTLIERS,
    seed=2026,
)

condition_error = np.max(np.abs(
    forecast_ppi_up_10["future_level_paths"][:, :, VARIABLES.index("ppi_energy")]
    - ppi_up_10[None, :]
))
print(f"Maximum absolute PPI condition error: {format_number(condition_error)}")

display(style_numeric_table(
    forecast_horizon_table(forecast_ppi_up_10, TARGET),
    caption=f"Conditional {TARGET} forecast — PPI Energy +10%",
))
plot_forecast_comparison(
    forecast_unconditional,
    forecast_ppi_up_10,
    historical_levels=levels,
    variable=TARGET,
    ylabel=UNITS[TARGET],
)
plt.show()

### 7.3 HICP solid fuels level and year-on-year inflation


In [ ]:
hicp_unconditional = forecast_hicp_component(
    forecast_unconditional,
    result,
    component=COMPONENT,
)
hicp_gas_up_10 = forecast_hicp_component(
    forecast_gas_up_10,
    result,
    component=COMPONENT,
)
hicp_ppi_up_10 = forecast_hicp_component(
    forecast_ppi_up_10,
    result,
    component=COMPONENT,
)

display(style_numeric_table(
    path_horizon_table(
        hicp_unconditional["future_hicp_level_paths"],
        hicp_unconditional["future_dates"],
        forecast_unconditional["last_calendar_date"],
    ),
    caption=f"HICP {SPEC['label']} index forecast",
))
display(style_numeric_table(
    path_horizon_table(
        hicp_unconditional["future_hicp_yoy_paths"],
        hicp_unconditional["future_dates"],
        forecast_unconditional["last_calendar_date"],
    ),
    caption=f"HICP {SPEC['label']} year-on-year inflation forecast (%)",
))

plot_hicp_forecast_fan(
    hicp_unconditional,
    kind="level",
    component_label=SPEC["label"],
)
plot_hicp_forecast_fan(
    hicp_unconditional,
    kind="yoy",
    component_label=SPEC["label"],
)
plt.show()

## 8. Tax assumptions and inflation impact

This component has **no separate tax-scenario layer in this model**. The target is the observed HICP solid fuels index itself, rather than a constructed pre-tax consumer price. VAT and excise are therefore not removed before estimation and cannot be re-attributed ex post without introducing an additional tax decomposition that is absent from the processed dataset/model specification.

The section is kept explicitly in the notebook so the architecture is consistent across all seven models: tax scenarios are implemented only for gas, electricity, petrol, diesel and liquid fuels, where the target is modelled pre-tax. For solid fuels, the usual HICP forecast above is already the full consumer-price forecast.


## 9. Structural analysis

The recursive ordering is wholesale natural gas, PPI Energy, then the HICP component. It therefore treats the commodity-price block as contemporaneously upstream of the consumer-price target. The alternative sign-restricted identification only imposes a positive own impact for each named shock.


In [ ]:
COMPONENT_SIGN_RESTRICTIONS = {
    "wholesale_gas_shock": {
        "natural_gas_wholesale": {0: +1},
    },
    "ppi_energy_shock": {
        "ppi_energy": {0: +1},
    },
    "solid_fuels_specific_shock": {
        TARGET: {0: +1},
    },
}

STRUCTURAL_REFERENCE_DATE = prep["balanced_end"]
STRUCTURAL_HORIZON = 12

irf_recursive = impulse_responses(
    result,
    identification="recursive",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    horizon=STRUCTURAL_HORIZON,
    shock_unit="structural_std",
    include_outlier_scale=False,
)
fevd_recursive = forecast_error_variance_decomposition(
    result,
    identification="recursive",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    horizon=STRUCTURAL_HORIZON,
    include_outlier_scale=False,
)
hd_recursive = historical_decomposition(
    result,
    identification="recursive",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    split_outlier_amplification=True,
)

print("Recursive identification")
print(f"Reference date: {pd.Timestamp(STRUCTURAL_REFERENCE_DATE).date()}")
print(f"HD maximum reconstruction error: {format_number(hd_recursive['max_reconstruction_error'])}")
plot_impulse_responses(irf_recursive, cumulative=False, units=UNITS)
plot_impulse_responses(irf_recursive, cumulative=True, units=UNITS)
plot_fevd(fevd_recursive)
for response in VARIABLES:
    plot_historical_decomposition(
        hd_recursive,
        response=response,
        last_obs=144,
        unit=UNITS[response],
    )
plt.show()

irf_sign = impulse_responses(
    result,
    identification="sign",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    sign_restrictions=COMPONENT_SIGN_RESTRICTIONS,
    horizon=STRUCTURAL_HORIZON,
    shock_unit="structural_std",
    include_outlier_scale=False,
    max_tries=20_000,
    seed=SEED,
)
fevd_sign = forecast_error_variance_decomposition(
    result,
    identification="sign",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    sign_restrictions=COMPONENT_SIGN_RESTRICTIONS,
    horizon=STRUCTURAL_HORIZON,
    include_outlier_scale=False,
    max_tries=20_000,
    seed=SEED,
)
hd_sign = historical_decomposition(
    result,
    identification="sign",
    reference_date=STRUCTURAL_REFERENCE_DATE,
    sign_restrictions=COMPONENT_SIGN_RESTRICTIONS,
    split_outlier_amplification=True,
    max_tries=20_000,
    seed=SEED,
)

print()
print("Sign-restricted identification")
print(f"Accepted posterior-draw share: {irf_sign['acceptance_rate']:.2%}")
print(f"Mean attempts per accepted draw: {irf_sign['mean_attempts_per_accepted_draw']:.2f}")
print(f"HD maximum reconstruction error: {format_number(hd_sign['max_reconstruction_error'])}")
plot_impulse_responses(irf_sign, cumulative=False, units=UNITS)
plot_impulse_responses(irf_sign, cumulative=True, units=UNITS)
plot_fevd(fevd_sign)
for response in VARIABLES:
    plot_historical_decomposition(
        hd_sign,
        response=response,
        last_obs=144,
        unit=UNITS[response],
    )
plt.show()

## 10. Diagnosis


### 10.1 Stationarity of transformed endogenous data


In [ ]:
stationarity_table = stationarity_tests(prep["balanced"])
display(style_numeric_table(
    stationarity_table,
    caption="Stationarity diagnostics",
))

### 10.2 MCMC effective sample size and autocorrelation


In [ ]:
diagnostic_table = mcmc_diagnostics(result)
display(style_numeric_table(
    diagnostic_table.sort_values("ESS"),
    caption="MCMC diagnostics",
))
plot_trace_grid(result)
plot_chain_acf_grid(result, nlags=60)
plt.show()

### 10.3 Stability


In [ ]:
display(style_numeric_table(
    stability_diagnostics(result).to_frame("value"),
    caption="VAR stability diagnostics",
))
plot_spectral_radius(result)
plt.show()

### 10.4 Posterior predictive checks


In [ ]:
ppc_summary, ppc_draws = posterior_predictive_statistics(
    result,
    n_draws=min(300, result["n_draws"]),
    seed=2027,
)
for equation in VARIABLES:
    equation_rows = ppc_summary.loc[
        [str(index).startswith(f"{equation}:") for index in ppc_summary.index]
    ]
    display(style_numeric_table(
        equation_rows,
        caption=f"Posterior predictive checks — {equation}",
    ))
    plot_posterior_predictive_check(ppc_summary, equation=equation)
    plt.show()

## 11. Validation checklist

Before freezing the solid fuels model, verify:

* the endogenous order is `natural_gas_wholesale`, `ppi_energy`, `hicp_solid_fuels`;
* the balanced sample has no interior missing month after absolute differencing;
* both conditional paths satisfy their imposed levels within numerical precision;
* the historical decomposition reconstructs observed changes within numerical precision;
* the HICP level paths and year-on-year paths are aligned to the same forecast dates;
* the Run ID and saved result directory are retained in the canonical HTML export.
